# Four-feature, all-speaker geometry baseline
F1, F2, F3 and duration only: no F0-based filtering or missingness feature. All 98 speakers and all 447,265 tokens. The four views and settings match the five-feature baseline. This is descriptive geometry, not held-out predictive evaluation. Change config_four.py, then set RESUME=None to launch a new run. Preprocessing is fitted to all observations for this descriptive analysis only.

In [ ]:
from pathlib import Path
import sys, runpy, subprocess, json
import numpy as np
import pandas as pd
from IPython.display import display, Image
ROOT = Path.cwd().parents[1] if Path.cwd().name == 'vowel_geometry' else Path.cwd()
assert (ROOT/'metadata_vowels_acoustics.csv').exists()
sys.path.insert(0, str(ROOT))
CONFIG_PATH = ROOT/'lab_projects/vowel_geometry/config_four.py'
RESUME = ROOT/'lab_projects/vowel_geometry/results_four/20260929_230640'
OLD_RUN = ROOT/'lab_projects/vowel_geometry/results/20260929_173507'
display(runpy.run_path(str(CONFIG_PATH))['CONFIG'])
print('Interpreter:', sys.executable)


In [ ]:
command = [sys.executable, str(ROOT/'lab_projects/vowel_geometry/analyze.py'), '--config', str(CONFIG_PATH)]
if RESUME is not None: command += ['--resume', str(RESUME)]
result = subprocess.run(command, cwd=ROOT, capture_output=True, text=True, check=True)
print(result.stdout)
RUN = Path(result.stdout.split('COMPLETE:')[-1].strip())


In [ ]:
from lab_projects.vowel_geometry.report import make_report
make_report(RUN)
summary = pd.read_csv(RUN/'summary.csv')
display(summary[summary.weighting.eq('macro')].pivot(index='view', columns='metric', values='mean'))
for name in ['ground_truth', 'cluster_agreement', 'speaker_variation', 'phoneme_variation', 'pca']:
    display(Image(filename=str(RUN/'figures'/f'{name}.png')))


In [ ]:
from lab_projects.vowel_geometry.compare_features import compare
comparison = compare(OLD_RUN, RUN)
display(comparison[comparison.metric.isin(['truth_silhouette','ari','nmi','ami'])])
metrics = pd.read_csv(RUN/'metrics.csv')
assert len(metrics) == 315 and metrics.seed.nunique() == 3
assert (metrics.groupby('view').n.sum() == 447265*3).all()
assert np.load(RUN/'standardized.npy').shape == (447265,4)
assert not metrics.fit_hit_max_iter.any()
print('Matched-design checks passed. Reports:', RUN/'REPORT.html', RUN/'COMPARISON.html')
